In [ ]:
import geopandas as gpd
import pandas as pd
import pyogrio
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Ruta del GeoPackage
gpkg_path = Path("../data/matriz_barrios_unificada.gpkg")

# Consultar las capas
layers = pyogrio.list_layers(gpkg_path)

display(pd.DataFrame(
    layers,
    columns=["Nombre de capa", "Tipo de geometría"]
))

In [ ]:
layer_name = layers[0][0]

gdf = gpd.read_file(
    gpkg_path,
    layer=layer_name,
    engine="pyogrio"
)

print("Dimensiones:", gdf.shape)
print("\nSistema de coordenadas:", gdf.crs)

print("\nColumnas:")
print(gdf.columns.tolist())

display(gdf.head())

In [ ]:
import plotly.express as px

# Crear identificador único para cada sector
gdf = gdf.reset_index(drop=True).copy()
gdf["sector_id"] = gdf.index.astype(str)

# Revisar la variable de densidad
print(gdf["densidad_airbnb_km2"].describe())

print(
    "\nValores faltantes:",
    gdf["densidad_airbnb_km2"].isna().sum()
)

print(
    "Valores iguales a cero:",
    (gdf["densidad_airbnb_km2"] == 0).sum()
)

In [ ]:
# Crear GeoJSON desde los polígonos
geojson = gdf.__geo_interface__

# Calcular centro aproximado del mapa
points = gdf.geometry.representative_point()

center_lat = points.y.mean()
center_lon = points.x.mean()

# Crear mapa coroplético
fig = px.choropleth_map(
    gdf,
    geojson=geojson,
    locations="sector_id",
    featureidkey="properties.sector_id",
    color="densidad_airbnb_km2",
    color_continuous_scale=[
        "#DCEAF7",
        "#A9CBE8",
        "#6D9FCC",
        "#376B9C",
        "#163B65"
    ],
    map_style="carto-darkmatter",
    center={
        "lat": center_lat,
        "lon": center_lon
    },
    zoom=9,
    opacity=0.85,
    hover_name="barrio",
    hover_data={
        "localidad": True,
        "densidad_airbnb_km2": ":.2f",
        "num_listings_airbnb": True,
        "sector_id": False
    },
    labels={
        "densidad_airbnb_km2": "Airbnb por km²",
        "localidad": "Localidad",
        "num_listings_airbnb": "Anuncios"
    }
)

fig.update_layout(
    title="Densidad de alojamientos Airbnb por sector",
    paper_bgcolor="#1B2638",
    font_color="white",
    margin=dict(l=0, r=0, t=45, b=0),
    height=650,
    coloraxis_colorbar=dict(
        title="Anuncios/km²"
    )
)

fig.show()

In [ ]:
# Buscar registros cuya localidad sea Chapinero
chapinero = gdf[
    gdf["localidad"].astype(str).str.contains(
        "chapinero",
        case=False,
        na=False
    )
]

print("Registros de Chapinero:", len(chapinero))

print("\nBarrios disponibles:")
display(
    chapinero[["localidad", "barrio"]]
    .drop_duplicates()
    .sort_values("barrio")
)

In [ ]:
seleccion = gdf[
    (gdf["localidad"] == "CHAPINERO") &
    (gdf["barrio"] == "LOS ROSALES")
].copy()

print("Registros encontrados:", len(seleccion))

display(
    seleccion[
        [
            "localidad",
            "barrio",
            "densidad_airbnb_km2",
            "num_listings_airbnb",
            "v_ref_2026"
        ]
    ]
)

In [ ]:
# Identificar el sector seleccionado
gdf["seleccionado"] = (
    (gdf["localidad"] == "CHAPINERO") &
    (gdf["barrio"] == "LOS ROSALES")
)

# Crear GeoJSON actualizado
geojson = gdf.__geo_interface__

# Crear mapa base
fig = px.choropleth_map(
    gdf,
    geojson=geojson,
    locations="sector_id",
    featureidkey="properties.sector_id",
    color="densidad_airbnb_km2",
    color_continuous_scale=[
        "#DCEAF7",
        "#A9CBE8",
        "#6D9FCC",
        "#376B9C",
        "#163B65"
    ],
    map_style="carto-darkmatter",
    center={
        "lat": center_lat,
        "lon": center_lon
    },
    zoom=10,
    opacity=0.85,
    hover_name="barrio",
    hover_data={
        "localidad": True,
        "densidad_airbnb_km2": ":.2f",
        "num_listings_airbnb": True,
        "sector_id": False
    }
)

# Agregar el sector seleccionado en rojo
fig.add_trace(
    go.Choroplethmap(
        geojson=geojson,
        locations=gdf.loc[
            gdf["seleccionado"], "sector_id"
        ],
        z=[1] * gdf["seleccionado"].sum(),
        featureidkey="properties.sector_id",
        colorscale=[
            [0, "#EF4444"],
            [1, "#EF4444"]
        ],
        showscale=False,
        marker_opacity=0.95,
        marker_line_color="#FFFFFF",
        marker_line_width=1.5,
        hovertemplate=(
            "<b>LOS ROSALES</b><br>"
            "Localidad: Chapinero<br>"
            "Anuncios Airbnb: 46<br>"
            "Densidad: 111,99 anuncios/km²"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Densidad de alojamientos Airbnb por sector",
    paper_bgcolor="#1B2638",
    font_color="white",
    margin=dict(l=0, r=0, t=45, b=0),
    height=650,
    coloraxis_colorbar=dict(
        title="Anuncios/km²"
    )
)

fig.show()

In [ ]:
from map_view import create_map

fig = create_map()

fig.show()